# K=64 headline, 60 -> 120 epochs (continuation)

Resumes the 60-epoch K=64 headline (0.98363 AUC, 1140.8 rejection at 30%%
signal efficiency, final validation loss 0.16276) from its epoch-60
checkpoint, embedded here, to 120 epochs. The cosine schedule is re-anchored
on the 120-epoch horizon, so the learning rate restarts at 1.5e-3.
Decision rule (plan_2026-10-10 §2), on the final weights against epoch 60:
dAUC >= +0.0003 with a lower final validation loss -> longer training pays;
dAUC <= +0.0001 -> training length exhausted near 60 epochs.

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True,
                     text=True).stdout.strip())

In [ ]:
import base64, io, tarfile, pathlib
CODE_B64 = "..."
tarfile.open(fileobj=io.BytesIO(base64.b64decode(CODE_B64)), mode="r:gz").extractall("/kaggle/working/repo")
print(sorted(p.name for p in pathlib.Path("/kaggle/working/repo").iterdir()))

In [ ]:
import subprocess, sys

def run(args, cwd="/kaggle/working/repo"):
    """Run a repo module, stream nothing, return (ok, tail-of-output)."""
    r = subprocess.run([sys.executable, "-m"] + args, cwd=cwd,
                       capture_output=True, text=True)
    out = r.stdout[-4000:]
    if r.returncode != 0:
        out += "\n--- stderr ---\n" + r.stderr[-3000:]
    print(out)
    return r.returncode == 0

In [ ]:
import subprocess, sys, time

def ensure_torch_for_this_gpu():
    import torch
    if not torch.cuda.is_available():
        print("no CUDA; nothing to fix")
        return
    cap = torch.cuda.get_device_capability(0)
    tag = "sm_%d%d" % cap
    if tag in torch.cuda.get_arch_list():
        print("torch", torch.__version__, "already supports", tag)
        return
    print("torch", torch.__version__, "lacks", tag,
          "- installing a compatible build")
    t0 = time.perf_counter()
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "torch==2.5.1", "--index-url",
                        "https://download.pytorch.org/whl/cu121"],
                       capture_output=True, text=True)
    print("pip rc", r.returncode, "in %.0f s" % (time.perf_counter() - t0))
    if r.returncode:
        print(r.stderr[-2000:])
        raise SystemExit("could not install a GPU-compatible torch")

ensure_torch_for_this_gpu()
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "torchdiffeq"],
               check=True)

In [ ]:
import glob, pathlib
cands = glob.glob("/kaggle/input/**/top_tagging_train.npz", recursive=True)
assert cands, "attach the K=128 data-prep kernel output"
DATA = str(pathlib.Path(cands[0]).parent)
TAG = 'headline120'
OUT = "/kaggle/working/results_k64/" + TAG
CKPT_DIR = "/kaggle/working/checkpoints/" + TAG
BASE = ['--cache-dir', 'DATA', '--representation', 'constituents', '--canonical-splits', '--normalize', 'global', '--device', 'cuda', '--dtype', 'float32', '--batch-size', '256', '--n-constituents', '64', '--eval-chunk-size', '1024', '--models', 'so3c_message_set', '--rounds', '3', '--beams', '--channels', '8', '--compile']
BASE[BASE.index("DATA")] = DATA
pathlib.Path(OUT).mkdir(parents=True, exist_ok=True)
pathlib.Path(OUT, "bundle_sha256.txt").write_text('408f9b823833' + chr(10))
print("data:", DATA, "| bundle", '408f9b823833')

In [ ]:
# On-card guard: the forward pass must be the published function here too.
import subprocess, sys
g = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider",
                    "tests/test_so3c_models.py", "-k", "golden_cuda"],
                   cwd="/kaggle/working/repo", capture_output=True, text=True)
print(g.stdout[-1200:], g.stderr[-800:])
assert "1 passed" in g.stdout, "golden guard did not pass on the card"

In [ ]:
# The embedded epoch-60 checkpoint: write it where --resume looks, verify it,
# and make sure this torch accepts its CUDA RNG state.
import base64, hashlib, pathlib, torch
CKPT_B64 = "..."
raw = base64.b64decode(CKPT_B64)
assert hashlib.sha256(raw).hexdigest() == '36ce59cf9fb83618096953cd0ab1810e3bd290b37ed6786daf2490f42402601c', "checkpoint corrupted in transit"
ck_path = pathlib.Path(CKPT_DIR) / "top_tagging_canonical__so3c_message_set__seed0.pt"
ck_path.parent.mkdir(parents=True, exist_ok=True)
ck_path.write_bytes(raw)
state = torch.load(ck_path, map_location="cpu", weights_only=False)
assert state["epoch"] == 60 and len(state["history"]) == 60, (state["epoch"], len(state["history"]))
assert abs(state["history"][-1]["val_loss"] - 0.16276) < 1e-5, state["history"][-1]["val_loss"]
print("checkpoint ok: epoch", state["epoch"], "| val loss %.5f" % state["history"][-1]["val_loss"],
      "| torch here", torch.__version__)
try:
    torch.cuda.set_rng_state(state["cuda_rng"].cpu(), 0)
    print("CUDA RNG state from torch 2.10 accepted")
except Exception as exc:
    print("CUDA RNG state rejected (%s); dropping it from the copy" % str(exc)[:200])
    state["cuda_rng"] = None
    torch.save(state, ck_path)
del state, raw, CKPT_B64

In [ ]:
# Smoke: one epoch on 20k jets with the real configuration (a fresh model,
# separate directories), to prove the data path and the compiled model
# before the session commits hours.
import glob
run(["benchmarks.run_top_tagging"] + BASE +
    ["--epochs", "1", "--max-train-samples", "20000", "--seed", "0",
     "--results-dir", "/kaggle/working/smoke"])
assert glob.glob("/kaggle/working/smoke/*.json"), "smoke run wrote no result"

In [ ]:
ok = run(["benchmarks.run_top_tagging"] + BASE +
         ["--epochs", "120", "--seed", "0", "--keep-best", "--resume",
          "--results-dir", OUT, "--ckpt-dir", CKPT_DIR,
          "--max-seconds", "39000"])
assert ok, "continuation run failed"

In [ ]:
import json, glob
REF_AUC, REF_REJ, REF_VL = (0.98363, 1140.8, 0.16276)
r = json.load(open(glob.glob(OUT + "/*seed0.json")[0]))
t, h = r["test_metrics"], r["history"]
print("epochs run %d of 120, session-cumulative wall clock %.2f h, env %s"
      % (r["epochs_run"], r["walltime_sec"] / 3600, r.get("env")))
if r["epochs_run"] < 120:
    print("INCOMPLETE: stopped on the session cap; push the finish kernel")
print("final weights:  AUC %.5f  rej@0.3 %.1f  rej@0.5 %.1f  final val loss %.5f"
      % (t["test_auc"], t["bg_rej_30"], t.get("bg_rej_50", float("nan")), h[-1]["val_loss"]))
b = r.get("test_metrics_best_val")
if b:
    print("best-val weights (epoch %d, val loss %.5f; epochs 61-120 only):  AUC %.5f  rej@0.3 %.1f"
          % (r["best_val_epoch"], r["best_val_loss"], b["test_auc"], b["bg_rej_30"]))
print()
print("validation curve")
for e in h:
    if e["epoch"] in (55, 60) or (e["epoch"] > 60 and e["epoch"] % 5 == 0) or e["epoch"] == 61:
        print("  epoch %3d  train %.5f  val %.5f  val_acc %.5f"
              % (e["epoch"], e["train_loss"], e["val_loss"], e["val_acc"]))
d_auc = t["test_auc"] - REF_AUC
print()
print("against epoch 60 of the same model (%.5f / %.1f / val %.5f): dAUC %+.5f, d rej %+.1f, d val loss %+.5f"
      % (REF_AUC, REF_REJ, REF_VL, d_auc, t["bg_rej_30"] - REF_REJ, h[-1]["val_loss"] - REF_VL))
if r["epochs_run"] == 120:
    if d_auc >= 0.0003 and h[-1]["val_loss"] < REF_VL:
        print("DECISION (plan 2.4): longer training pays at K=64")
    elif d_auc <= 0.0001:
        print("DECISION (plan 2.4): training length exhausted near 60 epochs; close")
    else:
        print("DECISION (plan 2.4): unresolved; record it")